---
title: "Part 2: Language Core (Control Flow & Comprehensions)"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/02-control-flow.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/02-control-flow.ipynb)

**DS-MLOps Python Foundations**

**Python 3.12+ | Author: Anthony Faustine**

Your training loop runs 50 epochs. Epoch 47 fails because a score came in as `None`. The loop should have skipped it. It didn't -- because no one told it what to do when the check fails. Control flow is how you teach code to make decisions.

This notebook covers four constructs: `if` for decisions, `for` for known collections, `while` for indefinite repetition, and comprehensions for one-line transformations. Same scenario as Part 1: student scores, course enrollments, and experiment logs.

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).

::: {.callout-note collapse="true" icon=false}
## Before You Begin

This notebook assumes you have completed Part 1 (`01-python-core.ipynb`). Later cells use variables and concepts from there. If something looks unfamiliar, open Part 1 and run the relevant section first.
:::

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

By the end of Part 2 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Write `if` / `elif` / `else` and structural `match` / `case` | Sec. 1 |
| 2 | Replace manual index counters with `for`, `enumerate`, and `zip` | Sec. 2 |
| 3 | Use `while`, `break`, and `continue` for indefinite loops | Sec. 3 |
| 4 | Replace loops with list, dict, set, and generator comprehensions | Sec. 4 |
:::


Before diving into each construct, here is the map. Python gives you four tools for controlling execution -- each answers a different question about what runs and when.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import matplotlib.pyplot as plt

BG = "#F8FAFC"
BLUE = "#0369A1"
GREEN = "#059669"
PURPLE = "#7C3AED"
AMBER = "#B45309"

configs = [
    (0, "if / elif / else", "Choose one branch\nbased on a condition", "#DBEAFE", BLUE),
    (1, "for loop", "Repeat once per item\nin a known collection", "#D1FAE5", GREEN),
    (2, "while loop", "Repeat until a condition\nbecomes False", "#EDE9FE", PURPLE),
    (3, "comprehension", "Transform a collection\nin one expression", "#FEF3C7", AMBER),
]

fig, ax = plt.subplots(figsize=(9, 2.4))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
ax.set_xlim(-0.1, 4.1)
ax.set_ylim(-0.05, 1.05)
ax.axis("off")

for col, name, desc, bg, border in configs:
    x = col + 0.5
    ax.add_patch(
        FancyBboxPatch(
            (col + 0.06, 0.08),
            0.88,
            0.84,
            boxstyle="round,pad=0.03",
            facecolor=bg,
            edgecolor=border,
            linewidth=2,
            zorder=2,
        )
    )
    ax.text(
        x,
        0.72,
        name,
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold",
        color=border,
        fontfamily="monospace",
        zorder=3,
    )
    ax.text(x, 0.38, desc, ha="center", va="center", fontsize=9, color="#374151", linespacing=1.5, zorder=3)

ax.set_title(
    "Python Control Flow: Four Constructs at a Glance", fontsize=12, fontweight="bold", color="#0F172A", pad=10
)
plt.tight_layout()
plt.show()

## 1. Control Flow: if / elif / else & match / case

So far every cell runs its lines from top to bottom, once, in order. **Control flow** lets you change that:
- `if / elif / else`: run one branch based on a condition
- `match / case`: route structured data to different handlers (Python 3.10+)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: match / case (Python 3.10+)</span><br><br>
Structural pattern matching goes beyond simple equality checks. It can match on the <b>shape of data</b>, destructuring dicts, lists, and class instances in the same step. Use it when branching on the shape or value of structured data, not just numeric thresholds.
</div>

In [ ]:
def classify_grade(score: float) -> str:
    """Return letter grade for a numeric score."""
    if score >= 90:
        return "A: Excellent"
    elif score >= 80:
        return "B: Good"
    elif score >= 70:
        return "C: Satisfactory"
    elif score >= 60:
        return "D: Needs improvement"
    else:
        return "F: See instructor"

Test the function across the full grade range:

In [ ]:
for s in [95.0, 83.5, 71.0, 62.0, 45.0]:
    print(f"  {s:5.1f} -> {classify_grade(s)}")

# Ternary expression: one-liner for simple binary choices
score = 87.0
status = "pass" if score >= 70 else "fail"
band = "high" if score >= 90 else ("mid" if score >= 70 else "low")
print(f"\n{score} -> {status}, {band}")

### match / case: Structural Pattern Matching (Python 3.10+)
`match` goes beyond simple equality checks. It can destructure **the shape of data**, extracting values from dicts and lists in one step. Define the routing function first:

In [ ]:
# match / case: structural pattern matching (Python 3.10+)
def process_event(event: dict[str, object]) -> str:
    """Route a training event to the right handler."""
    match event:
        case {"type": "epoch", "epoch": e, "loss": l} if float(str(l)) < 0.05:
            return f"Epoch {e}: converged (loss={l:.3f})"
        case {"type": "epoch", "epoch": e, "loss": l}:
            return f"Epoch {e}: loss={l:.3f}"
        case {"type": "error", "message": msg}:
            return f"ERROR: {msg}"
        case {"type": t}:
            return f"Unhandled event type: {t!r}"
        case _:
            return "Malformed event"

Run a variety of event shapes through the dispatcher to see each `case` arm triggered. The `case _:` arm is a catch-all that always matches:

In [ ]:
events: list[dict[str, object]] = [
    {"type": "epoch", "epoch": 1, "loss": 0.823},
    {"type": "epoch", "epoch": 20, "loss": 0.041},
    {"type": "error", "message": "OOM on GPU 0"},
    {"type": "checkpoint"},
    {"status": "idle"},
]

for ev in events:
    print(process_event(ev))

### Sequence Patterns in match / case

Sequence patterns match lists and tuples by position. `case [first, *rest]:` binds the first element to `first` and the remainder to `rest`. This is particularly useful in data pipelines where you process variable-length records.

In [ ]:
def describe_scores(record: list) -> str:
    match record:
        case []:
            return "no scores"
        case [only]:
            return f"single score: {only}"
        case [first, second]:
            return f"two scores: {first} and {second}"
        case [first, *rest] if first >= 90:
            return f"high-starter ({first}) with {len(rest)} more scores"
        case [first, *rest]:
            return f"started at {first}, {len(rest)} more scores"


for r in [[], [85], [72, 88], [95, 80, 77], [60, 70, 75, 80]]:
    print(f"{r!r:30s} -> {describe_scores(r)}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Match on HTTP-style Status Codes</span><br><br>
<b>Goal:</b> Write a <code>describe_status(code)</code> function using <code>match/case</code> that returns a short description.
<pre style='background:#FCE8DA;padding:10px;border-radius:4px;font-size:0.9em'>describe_status(200)  -> '200 OK'
describe_status(404)  -> '404 Not Found'
describe_status(500)  -> '500 Server Error'
describe_status(301)  -> '3xx Redirect'
describe_status(999)  -> 'Unknown code'</pre>
<b>Hint:</b> Use <code>case 2xx</code> patterns are not valid. Use guard conditions instead: <code>case c if 200 <= c < 300</code>.
</div>

In [ ]:
def describe_status(code: int) -> str:
    """Return a short description for an HTTP-style status code."""
    match code:
        case _:
            return "unknown"  # TODO: replace with specific case patterns


for c in [200, 404, 500, 301, 999]:
    print(describe_status(c))

## 2. Control Flow: for Loops

A **`for` loop** repeats a block of code once for each item in a collection. It is the primary tool for processing datasets, running training epochs, and iterating over files.

```python
for score in [78, 85, 92]:   # repeat once per score
    print(score)              # output: 78, then 85, then 92
```

The indented block (4 spaces) is the **loop body**: it runs once per item.

Python `for` loops iterate over any **iterable**. The built-ins `range()`, `enumerate()`, and `zip()` cover the most common patterns in data work.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: enumerate and zip over manual indexing</span><br><br>
<code>for i in range(len(items))</code> is a red flag. Use <code>enumerate(items)</code> when you need both the index and the value, and <code>zip(a, b)</code> when you need to step two collections in lockstep. Both are lazy: they generate pairs on demand without building a temporary list.
</div>

In [ ]:
# range(start, stop, step): generates integers lazily (no list in memory)
MAX_EPOCHS: int = 5
loss: float = 1.0

for epoch in range(1, MAX_EPOCHS + 1):
    loss *= 0.75
    print(f"  Epoch {epoch}/{MAX_EPOCHS}  loss={loss:.4f}")

`enumerate()` pairs each element with its index, counting from `start=1` by default (or any integer you choose), eliminating the need for manual `i += 1` counters:

In [ ]:
# enumerate(): loop with automatic index; avoids manual counter variables
students: list[str] = ["Alice", "Carol", "Dan", "Bob"]

print("Leaderboard:")
for rank, name in enumerate(students, start=1):
    print(f"  #{rank}  {name}")

`zip()` stitches two or more iterables together element-by-element. Pairs stop when the **shortest** input is exhausted. Build a `dict` from two parallel lists using `dict(zip(keys, values))`:

In [ ]:
# zip(): iterate two or more iterables in lockstep
# strict=True raises ValueError if the iterables have different lengths
names: list[str] = ["Alice", "Bob", "Carol"]
scores: list[float] = [92.0, 74.5, 88.0]

print("Score sheet:")
for name, score in zip(names, scores, strict=True):
    grade = "pass" if score >= 70 else "fail"
    print(f"  {name:<8} {score:5.1f}  {grade}")

# Build a dict from two parallel lists
metric_names: list[str] = ["accuracy", "precision", "recall"]
metric_vals: list[float] = [0.923, 0.911, 0.934]
report: dict[str, float] = dict(zip(metric_names, metric_vals, strict=True))
print()
print(f"Report: {report}")

### tqdm: Progress Bars for Long Loops

When a loop processes thousands of files or training examples, you need to know how long it will take. `tqdm` wraps any iterable and displays a live progress bar with elapsed time, rate, and ETA, with zero code changes to the loop body:

```python
pip install tqdm   # if not already installed
```

In [ ]:
from tqdm import tqdm

# Wrap any iterable with tqdm() - the loop body is unchanged
scores: list[float] = []
for i in tqdm(range(1_000), desc="Simulating scores", unit="rec"):
    scores.append(50 + (i % 50))  # dummy computation

print(f"Generated {len(scores)} scores, mean = {sum(scores) / len(scores):.1f}")

# tqdm also works with enumerate and zip
labels: list[str] = ["pass" if s >= 70 else "fail" for s in tqdm(scores, desc="Labelling", leave=False)]
print(f"pass rate: {labels.count('pass') / len(labels):.1%}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Training Log with enumerate and zip</span><br><br>
<b>Goal:</b> Print a formatted training summary using <code>enumerate</code> and <code>zip</code>.<br><br><b>Setup:</b><br><code>epochs = [1, 2, 3, 4, 5]</code><br><code>train_loss = [0.95, 0.72, 0.54, 0.41, 0.33]</code><br><code>val_loss   = [0.98, 0.76, 0.58, 0.47, 0.39]</code><br><br><b>Steps:</b><br>1. Use <code>zip</code> to pair <code>train_loss</code> and <code>val_loss</code> and iterate them together.<br>2. Use <code>enumerate</code> (starting from 1) to add an epoch counter.<br>3. Print each row as: <code>Epoch 1/5 | train=0.950 val=0.980</code>.<br>4. After the loop, print the epoch where <code>val_loss</code> was lowest.<br><br><b>Expected output (last line):</b> <code>Best epoch: 5 (val_loss=0.390)</code>
</div>

## 3. Control Flow: while, break, continue

A **`while` loop** repeats a block as long as a condition is `True`. Unlike `for` (which iterates a fixed collection), `while` runs an **indefinite** number of times until either the condition becomes `False` or a `break` statement is hit.

```python
loss = 1.0
while loss > 0.05:    # keep running until loss is small enough
    loss *= 0.7       # shrink loss by 30% each iteration
```

Use `while` when you do not know in advance how many iterations are needed: waiting for convergence, retrying a failing operation, or consuming a data stream.

- `break`: exit the loop immediately
- `continue`: skip the rest of this iteration
- `else` on a loop: runs **only** if no `break` was hit

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: while is for indefinite loops; for is for known collections</span><br><br>
Use <code>for</code> when you know the collection to iterate. Use <code>while</code> when you don't know how many iterations you need: training until convergence, reading until EOF, retrying until success. Every <code>while</code> loop needs either a condition that eventually becomes <code>False</code> or an explicit <code>break</code> -- otherwise it runs forever.
</div>

In [ ]:
# while: train until convergence or budget exhausted
loss: float = 1.0
epoch: int = 0
MAX_EPOCHS: int = 30
THRESHOLD: float = 0.05

while loss > THRESHOLD and epoch < MAX_EPOCHS:
    loss *= 0.7
    epoch += 1

print(f"Stopped at epoch {epoch}: loss={loss:.4f}")
print(f"Converged: {loss <= THRESHOLD}")

### break and continue
`break` exits the innermost loop immediately. Use it when a sentinel value or error condition means further iteration is pointless:

In [ ]:
# break: exit the loop immediately when a sentinel is found
readings: list[float | None] = [36.5, 36.9, 37.4, None, 38.1, 37.8]
clean: list[float] = []

for r in readings:
    if r is None:
        print("Sensor error : stopping collection")
        break
    clean.append(r)

print(f"Clean readings: {clean}")

`continue` skips the rest of the current iteration and jumps to the next one. Ideal for filtering bad data without a nested `if/else`. The `else` clause on a loop runs only if no `break` occurred:

In [ ]:
# continue: skip the rest of this iteration and move to the next
raw: list[object] = [85.0, "n/a", None, 92.0, "", 78.5, -1.0, 95.0]
valid: list[float] = []

for item in raw:
    if not isinstance(item, int | float) or float(str(item)) < 0:
        continue  # skip bad items
    valid.append(float(str(item)))

print(f"Valid scores: {valid}")

# loop else: runs only when the loop was NOT exited via break
required_fields: list[str] = ["name", "gpa", "major"]
record: dict[str, str] = {"name": "Alice", "gpa": "3.95", "major": "CS"}

for field in required_fields:
    if field not in record:
        print(f"Missing required field: {field!r}")
        break
else:
    print("All required fields present")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Early-Stopping Loop</span><br><br>
<b>Goal:</b> Implement a training loop that stops when the validation loss stops improving.<br><br><b>Setup:</b><br><code>val_losses = [0.95, 0.80, 0.65, 0.61, 0.60, 0.61, 0.63, 0.65]</code><br><br><b>Steps:</b><br>1. Use a <code>while</code> loop to iterate through <code>val_losses</code>.<br>2. Track the best loss seen so far. When the current loss is greater than the best, increment a <code>patience</code> counter; when it improves, reset <code>patience</code> to 0.<br>3. <code>break</code> when <code>patience</code> reaches 2 (two epochs without improvement).<br>4. Print the epoch at which training stopped and the best val_loss achieved.<br><br><b>Expected output:</b> <code>Stopped at epoch 7 | best val_loss=0.600</code>
</div>

## 4. Comprehensions

A **comprehension** builds a new collection by transforming or filtering an existing one, all in a single expression. It replaces the verbose `for` + `.append()` pattern:

```python
# Loop version (3 lines):
squares = []
for n in range(5):
    squares.append(n ** 2)   # [0, 1, 4, 9, 16]

# Comprehension (1 line, identical result):
squares = [n ** 2 for n in range(5)]
```

Comprehensions are faster than equivalent loops and are considered idiomatic Python.

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Concise, Readable Collection Construction</span><br><br>
Comprehensions build new collections by <b>transforming or filtering</b> an iterable in a single expression. They are faster than equivalent <code>for</code> + <code>.append()</code> loops and are idiomatic Python.

<table style='margin-top:8px;border-collapse:collapse'>
<tr><td style='padding:4px 12px;font-family:monospace'>[expr for x in it if cond]</td><td style='padding:4px 12px'>list</td></tr>
<tr><td style='padding:4px 12px;font-family:monospace'>{k: v for x in it if cond}</td><td style='padding:4px 12px'>dict</td></tr>
<tr><td style='padding:4px 12px;font-family:monospace'>{expr for x in it if cond}</td><td style='padding:4px 12px'>set</td></tr>
<tr><td style='padding:4px 12px;font-family:monospace'>(expr for x in it if cond)</td><td style='padding:4px 12px'>generator (lazy, no list in memory)</td></tr>
</table>
</div>

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Comprehensions are for simple transformations; loops are for complex logic</span><br><br>
A list comprehension <code>[f(x) for x in items if cond(x)]</code> is readable when the filter and transform are each one expression. When you need multiple steps, early returns, or side effects, write a <code>for</code> loop. Nested comprehensions beyond two levels hurt readability -- that's the signal to refactor.
</div>

In [ ]:
raw_scores: list[float] = [78.0, 85.5, 92.0, 88.5, 95.0, 67.0, 81.0]

# Transform: min-max normalise to [0, 1]
lo, hi = min(raw_scores), max(raw_scores)
normed: list[float] = [(s - lo) / (hi - lo) for s in raw_scores]
print(f"Normalised: {[round(n, 2) for n in normed]}")

# Filter: keep only passing scores
passing: list[float] = [s for s in raw_scores if s >= 70]
print(f"Passing   : {passing}")

# Filter + transform: label each score
labels: list[str] = [f"{s:.0f} (pass)" if s >= 70 else f"{s:.0f} (FAIL)" for s in raw_scores]
print(f"Labelled  : {labels}")

A two-clause comprehension flattens a nested collection. Read `[s for batch in batches for s in batch]` left-to-right: "outer loop, inner loop, collect `s`":

In [ ]:
# Flatten a nested structure with a two-clause comprehension
batches: list[list[float]] = [[85.0, 91.0], [74.0, 88.5], [95.0, 79.0]]
flat: list[float] = [s for batch in batches for s in batch]
print(f"Flattened : {flat}")

### Dict, Set, and Generator Comprehensions
The `[...]` syntax extends to dicts (`{k: v for ...}`), sets (`{expr for ...}`), and lazy generators (`(expr for ...)`):

In [ ]:
students: list[dict[str, object]] = [
    {"name": "Alice", "score": 92.0, "major": "CS"},
    {"name": "Bob", "score": 74.5, "major": "Math"},
    {"name": "Carol", "score": 88.0, "major": "CS"},
    {"name": "Dan", "score": 61.0, "major": "Physics"},
]

# Dict comprehension: build a name -> score lookup
score_lookup: dict[str, float] = {str(s["name"]): float(str(s["score"])) for s in students}
print(f"Lookup : {score_lookup}")

# Dict comprehension with filter: honours students only
honours: dict[str, float] = {str(s["name"]): float(str(s["score"])) for s in students if float(str(s["score"])) >= 80}
print(f"Honours: {honours}")

Set comprehensions deduplicate automatically. Generator expressions compute values **lazily**: they use O(1) memory regardless of input size, making them ideal inside `sum()`, `any()`, and `all()`:

In [ ]:
students: list[dict[str, object]] = [
    {"name": "Alice", "score": 92.0, "major": "CS"},
    {"name": "Bob", "score": 74.5, "major": "Math"},
    {"name": "Carol", "score": 88.0, "major": "CS"},
    {"name": "Dan", "score": 61.0, "major": "Physics"},
]

# Set comprehension: unique majors
majors: set[str] = {str(s["major"]) for s in students}
print(f"Majors : {sorted(majors)}")

# Generator expression: lazy evaluation; ideal inside sum/any/all
total: float = sum(float(str(s["score"])) for s in students)
any_fail: bool = any(float(str(s["score"])) < 70 for s in students)
all_pass: bool = all(float(str(s["score"])) >= 60 for s in students)

print(f"Mean          : {total / len(students):.1f}")
print(f"Any fail (<70): {any_fail}")
print(f"All pass (>=60): {all_pass}")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Cohort Score Report</span><br><br>
<b>Goal:</b> Using a single comprehension for each, produce the outputs below from <code>records</code>.
<pre style='background:#FCE8DA;padding:10px;border-radius:4px;font-size:0.9em'>records = [
    {'name': 'Alice', 'scores': [88, 92, 85]},
    {'name': 'Bob',   'scores': [62, 70, 58]},
    {'name': 'Carol', 'scores': [91, 95, 89]},
]

# 1. List of averages (one float per student)
averages = [82.33, 63.33, 91.67]

# 2. Dict mapping name -> average (rounded to 2 dp)
avg_map = {'Alice': 88.33, 'Bob': 63.33, 'Carol': 91.67}

# 3. Set of unique student names who scored >= 80 average
top = {'Alice', 'Carol'}</pre>
</div>

In [ ]:
records: list[dict[str, object]] = [
    {"name": "Alice", "scores": [88, 92, 85]},
    {"name": "Bob", "scores": [62, 70, 58]},
    {"name": "Carol", "scores": [91, 95, 89]},
]

# TODO: 1. list of averages
averages: list[float] = ...

# TODO: 2. name -> average dict
avg_map: dict[str, float] = ...

# TODO: 3. set of names with average >= 80
top: set[str] = ...

print(f"averages: {averages}")
print(f"avg_map : {avg_map}")
print(f"top     : {top}")

`itertools.batched()` (Python 3.12) splits any iterable into fixed-size chunks without loading everything into memory. It replaces manual sliding-window loops in data pipelines.

In [ ]:
from itertools import batched

student_ids = list(range(1, 22))  # 21 students

for batch_num, batch in enumerate(batched(student_ids, 5), start=1):
    print(f"Batch {batch_num}: {list(batch)}")

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: batched() for Chunked Processing</span><br><br>
<code>batched(iterable, n)</code> yields tuples of at most <code>n</code> items. The last batch may be shorter. Use it for API rate-limiting, mini-batch gradient descent, or processing large CSV files in chunks: <code>for rows in batched(csv_reader, 1000): process(rows)</code>.
</div>

## Capstone: Monte Carlo Pi Estimation

This activity ties together everything from Part 1 and Part 2: variables, lists, for loops, random numbers, functions, and comprehensions, to estimate the value of π using a simulation technique called **Monte Carlo integration**.

### The idea

Imagine a unit circle (radius = 1) inscribed in a 2×2 square. A random point `(x, y)` with `x, y ∈ [−1, 1]` falls inside the circle if `x² + y² ≤ 1`.

The ratio of the circle's area to the square's area is π/4. If we throw millions of random points and count how many land inside the circle, the proportion converges to π/4, so `π ≈ 4 × (hits / total)`.

```
  ┌──────────────┐
  │    ·  ●  ·   │   ● inside circle  → hit
  │  ●       ●   │   · outside        → miss
  │    circle    │
  │  ●       ●   │
  │    ·  ●  ·   │
  └──────────────┘
   π/4 ≈ hits/total
```

This is a real technique used in finance, physics, and ML for problems that are too complex to solve analytically.

**Step 1:** Write a helper that checks whether a point is inside the unit circle:

In [ ]:
import math


def in_unit_circle(x: float, y: float) -> bool:
    """Return True if (x, y) lies inside the unit circle (radius = 1)."""
    return x**2 + y**2 <= 1.0

**Step 2:** Simulate random points and count how many land inside the circle. `random.seed()` makes results reproducible. Always set a seed before any simulation:

In [ ]:
import random

random.seed(42)  # fix seed for reproducibility

N_POINTS: int = 1_000_000
inside: int = sum(
    1
    for _ in range(N_POINTS)
    if in_unit_circle(random.uniform(-1, 1), random.uniform(-1, 1))  # noqa: S311
)

pi_estimate: float = 4 * inside / N_POINTS
print(f"Points       : {N_POINTS:,}")
print(f"Hits (inside): {inside:,}")
print(f"pi estimate  : {pi_estimate:.5f}")
print(f"math.pi      : {math.pi:.5f}")
print(f"Error        : {abs(pi_estimate - math.pi):.5f}")

**Step 3:** See how the estimate improves as `N` grows: the law of large numbers at work:

In [ ]:
import math
import random

random.seed(0)

for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
    hits = sum(
        1
        for _ in range(n)
        if in_unit_circle(random.uniform(-1, 1), random.uniform(-1, 1))  # noqa: S311
    )
    est = 4 * hits / n
    error = abs(est - math.pi)
    print(f"  n={n:>9,}  pi={est:.5f}  error={error:.5f}")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> What you just used</span><br><br>
<ul>
<li><b>Variables & types</b>: <code>N_POINTS: int</code>, <code>pi_estimate: float</code></li>
<li><b>Functions</b>: <code>in_unit_circle()</code> with type hints</li>
<li><b>for loop</b>: iterating N times, building results</li>
<li><b>Comprehension</b>: <code>sum(1 for _ in range(n) if ...)</code></li>
<li><b>random module</b>: <code>seed()</code> for reproducibility, <code>uniform()</code> for sampling</li>
<li><b>math module</b>: <code>math.pi</code> as ground truth</li>
</ul>
This exact pattern (sample randomly, count outcomes, estimate a ratio) appears in A/B testing, Bayesian inference, and reinforcement learning.
</div>

## Further Reading

| Resource | Why it matters |
|---|---|
| [PEP 636: Structural Pattern Matching](https://peps.python.org/pep-0636/) | Official tutorial for `match`/`case`, with worked examples from the Python core team |
| Ramalho, L. (2022). *Fluent Python*, 2nd ed. O'Reilly. | Chapter 10 covers pattern matching in depth, including class patterns and guards |
| [Real Python: Python `for` Loops](https://realpython.com/python-for-loop/) | Clear treatment of `enumerate`, `zip`, and the iterator protocol behind every loop |
| [Real Python: List Comprehensions](https://realpython.com/list-comprehension-python/) | When to use comprehensions vs explicit loops, and how to avoid making them unreadable |


## Summary

| Concept | Key rule |
|---|---|
| `match`/`case` | Structural pattern matching on values, dicts, lists (3.10+) |
| `enumerate` / `zip` | Always prefer these over manual index counters |
| `while` / `break` / `continue` | For indefinite loops, early exit, and skipping bad data |
| Comprehensions | `[expr for x in it if cond]`; use generators `(...)` inside `sum()` / `any()` / `all()` |

**Next:** `03-python-patterns.ipynb`, covering functions, lambdas, `*args`/`**kwargs`, dataclasses, modules, exception handling, and file I/O with `pathlib`.